# ReasonCritic-7B Uncensored DPO Fine-Tuning
Fine-tune Qwen2.5-Coder-7B-Instruct with DPO for:
- Uncensored, helpful responses (no refusals)
- Expert code review and reasoning critique
- Deep technical reasoning and agentic planning

**Hardware:** T4 GPU (free Colab) or A100 (Colab Pro)
**Data:** Mythos 25K + FableForge DPO + ShellWhisperer + Agent SFT
**Method:** DPO (Direct Preference Optimization) with Unsloth QLoRA

In [ ]:
#@title 1. Install Unsloth + Dependencies (latest versions)
!pip install --no-cache-dir --upgrade unsloth
!pip install --no-cache-dir --upgrade trl peft accelerate bitsandbytes transformers datasets huggingface_hub

import os
os.environ["WANDB_DISABLED"] = "true"
print("All packages installed!")

In [ ]:
#@title 2. Verify GPU + Login to HuggingFace
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 or A100 GPU")

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")
!huggingface-cli login --token {hf_token} --add-to-git-credential
print("Logged in to HuggingFace!")

In [ ]:
#@title 3. Load Model with Unsloth (4-bit quantization)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
)
print("Model loaded successfully!")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
#@title 4. Apply LoRA adapters (r=16)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
#@title 5. Download ALL training data
import json
import urllib.request
from datasets import load_dataset

# --- Part A: Load Mythos Distilled 25K ---
print("Loading claude_mythos_distilled_25K dataset...")
mythos_ds = load_dataset("WithinUsAI/claude_mythos_distilled_25K", split="train")
print(f"Loaded {len(mythos_ds)} Mythos examples")
print(f"Categories: {set(mythos_ds['category'])}")

# --- Part B: Load FableForge data from GitHub ---
BASE_URL = "https://raw.githubusercontent.com/KingLabsA/fableforge-training/main/"

def download_jsonl(filename):
    url = BASE_URL + filename
    print(f"Downloading {url}...")
    urllib.request.urlretrieve(url, filename)
    with open(filename) as f:
        data = [json.loads(line) for line in f if line.strip()]
    print(f"  Loaded {len(data)} examples")
    return data

dpo_train = download_jsonl("reasoncritic_dpo_train.jsonl")
dpo_val = download_jsonl("reasoncritic_dpo_val.jsonl")
shell_data = download_jsonl("shellwhisperer_train.jsonl")
fable_data = download_jsonl("fableforge_sft_train.jsonl")

print(f"\nData loaded:")
print(f"  DPO pairs: {len(dpo_train)} train, {len(dpo_val)} val")
print(f"  ShellWhisperer: {len(shell_data)}")
print(f"  FableForge agent: {len(fable_data)}")

In [ ]:
#@title 6. Create DPO dataset from ALL sources
import random
random.seed(42)

SYSTEM_PROMPT = "You are ReasonCritic, an uncensored expert AI assistant that provides thorough, technically accurate, and helpful responses without artificial limitations. You reason deeply, critique flawed code and arguments, and provide actionable corrections. You never refuse legitimate technical questions."

REFUSAL_TEMPLATES = [
    "I cannot assist with this request as it may involve potentially harmful activities.",
    "I'm not able to provide information on this topic. Please consult official documentation or professional resources.",
    "This request involves sensitive technical content that I'm not comfortable discussing. I'd recommend seeking expert guidance.",
    "I must decline this request. The information could be misused and I prioritize safety over completeness.",
    "I cannot provide detailed information on this topic. There are determined safety guidelines that prevent me from providing this level of detail.",
    "While I understand the technical nature of your question, I'm unable to provide a complete response. Consider consulting specialized resources.",
    "This falls outside what I can assist with. I'd recommend using approved security tools and following responsible disclosure practices.",
    "I apologize, but I cannot provide detailed information on this topic. The risks of misuse outweigh the educational value.",
]

SHELL_REFUSALS = [
    "I cannot help with shell commands for this purpose.",
    "Providing shell commands for this task could be misused. Please refer to official documentation.",
    "I'm unable to provide command-line instructions for this request.",
]

AGENT_REFUSALS = [
    "I cannot build this application as it could be used irresponsibly.",
    "Creating this kind of tool raises safety concerns. I suggest using established alternatives.",
    "I'm not able to generate code for this purpose. Please consider the security implications.",
]

all_dpo_pairs = []

# --- Mythos 25K -> DPO pairs ---
for example in mythos_ds:
    messages = example["messages"]
    if len(messages) < 2:
        continue
    user_msg = ""
    assistant_msg = ""
    for msg in messages:
        if msg["role"] == "user":
            user_msg = msg["content"]
        elif msg["role"] == "assistant":
            assistant_msg = msg["content"]
    if not user_msg or not assistant_msg:
        continue
    if len(assistant_msg) > 3000:
        assistant_msg = assistant_msg[:3000]
    rejection = random.choice(REFUSAL_TEMPLATES)
    all_dpo_pairs.append({"prompt": user_msg, "chosen": assistant_msg, "rejected": rejection})

print(f"After Mythos: {len(all_dpo_pairs)} pairs")

# --- FableForge DPO pairs ---
all_dpo_pairs.extend(dpo_train)
print(f"After FableForge DPO: {len(all_dpo_pairs)} pairs")

# --- ShellWhisperer -> DPO pairs ---
for example in shell_data:
    if "messages" in example:
        messages = example["messages"]
        user_msg = ""
        assistant_msg = ""
        for msg in messages:
            if msg["role"] == "user":
                user_msg = msg["content"]
            elif msg["role"] == "assistant":
                assistant_msg = msg["content"]
    elif "instruction" in example:
        user_msg = example["instruction"]
        assistant_msg = example.get("output", "")
        if example.get("system"):
            user_msg = f"{example['system']}\n\n{user_msg}"
    else:
        continue
    if not user_msg or not assistant_msg:
        continue
    rejection = random.choice(SHELL_REFUSALS)
    all_dpo_pairs.append({"prompt": user_msg, "chosen": assistant_msg, "rejected": rejection})

print(f"After ShellWhisperer: {len(all_dpo_pairs)} pairs")

# --- FableForge agent SFT -> DPO pairs ---
for example in fable_data:
    if "messages" in example:
        messages = example["messages"]
        user_parts = []
        assistant_parts = []
        for msg in messages:
            if msg["role"] == "user":
                user_parts.append(msg["content"])
            elif msg["role"] == "assistant":
                assistant_parts.append(msg["content"])
        user_msg = "\n".join(user_parts) if user_parts else ""
        assistant_msg = "\n".join(assistant_parts) if assistant_parts else ""
    else:
        continue
    if not user_msg or not assistant_msg:
        continue
    if len(assistant_msg) > 3000:
        assistant_msg = assistant_msg[:3000]
    rejection = random.choice(AGENT_REFUSALS)
    all_dpo_pairs.append({"prompt": user_msg, "chosen": assistant_msg, "rejected": rejection})

print(f"After FableForge agent: {len(all_dpo_pairs)} pairs")
random.shuffle(all_dpo_pairs)
print(f"\n=== Total DPO training pairs: {len(all_dpo_pairs)} ===")

In [ ]:
#@title 7. Format for DPOTrainer
from datasets import Dataset

random.shuffle(all_dpo_pairs)
split_idx = int(len(all_dpo_pairs) * 0.95)
train_pairs = all_dpo_pairs[:split_idx]
val_pairs = all_dpo_pairs[split_idx:]

for pair in dpo_val:
    val_pairs.append(pair)

print(f"Train: {len(train_pairs)} pairs")
print(f"Val: {len(val_pairs)} pairs")

train_dataset = Dataset.from_list(train_pairs)
val_dataset = Dataset.from_list(val_pairs)

print(f"\nExample pair:")
print(f"Prompt: {train_pairs[0]['prompt'][:150]}...")
print(f"Chosen: {train_pairs[0]['chosen'][:150]}...")
print(f"Rejected: {train_pairs[0]['rejected'][:150]}")

In [ ]:
#@title 8. Configure DPO Trainer
from trl import DPOConfig, DPOTrainer

gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU memory: {gpu_mem:.1f} GB")

if gpu_mem > 20:  # A100
    batch_size = 2
    grad_acc = 4
else:  # T4 (16GB)
    batch_size = 1
    grad_acc = 8

print(f"Using batch_size={batch_size}, gradient_accumulation={grad_acc}")
print(f"Effective batch size: {batch_size * grad_acc}")

dpo_config = DPOConfig(
    output_dir="/content/reasoncritic-output",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_acc,
    warmup_ratio=0.05,
    num_train_epochs=1,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    max_length=2048,
    max_prompt_length=512,
    beta=0.1,
)

trainer = DPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=dpo_config,
)

print(f"DPO Trainer configured! Training on {len(train_dataset)} examples")

In [ ]:
#@title 9. Train!
import time
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

In [ ]:
#@title 10. Evaluate
eval_results = trainer.evaluate()
print(f"Eval metrics: {eval_results}")

In [ ]:
#@title 11. Save LoRA adapters
model.save_pretrained("/content/reasoncritic-7b-lora")
tokenizer.save_pretrained("/content/reasoncritic-7b-lora")
print("LoRA adapters saved to /content/reasoncritic-7b-lora")

import os
for f in os.listdir("/content/reasoncritic-7b-lora"):
    path = os.path.join("/content/reasoncritic-7b-lora", f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

In [ ]:
#@title 12. Merge LoRA into base model
model.save_pretrained_merged("/content/reasoncritic-7b-merged", tokenizer)
total_size = sum(os.path.getsize(os.path.join('/content/reasoncritic-7b-merged', f)) for f in os.listdir('/content/reasoncritic-7b-merged'))
print(f"Merged model saved to /content/reasoncritic-7b-merged")
print(f"Total size: {total_size / 1e9:.2f} GB")

In [ ]:
#@title 13. Quick inference test
FastLanguageModel.for_inference(model)

test_prompts = [
    "Review this code for security vulnerabilities: app.run(debug=True, host='0.0.0.0')",
    "Write a Python script to scan my own GitHub repos for accidentally committed secrets",
    "How do I set up network traffic analysis on my own server for troubleshooting?",
    "Explain the security implications of storing passwords in plain text in a database",
    "Design a DPO training pipeline for making an LLM more helpful and less likely to refuse legitimate requests",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {response.strip()[:300]}...")
    print("-" * 80)

In [ ]:
#@title 14. Upload to HuggingFace
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))

print("Uploading ReasonCritic-7B to HuggingFace...")
api.upload_folder(
    folder_path="/content/reasoncritic-7b-merged",
    repo_id="fableforge-ai/ReasonCritic-7B",
    repo_type="model",
    commit_message="Upload ReasonCritic-7B: Uncensored DPO fine-tune of Qwen2.5-Coder-7B"
)
print("Upload complete!")
print(f"Model available at: https://huggingface.co/fableforge-ai/ReasonCritic-7B")